In [1]:
# ==============================

In [2]:
import pandas as pd
import sqlite3

# create database
conn = sqlite3.connect("olist.db")

In [3]:
base_path = r"C:\Users\Yugadhi\OneDrive\Desktop\coding ninjas\SQL\New folder\OlistData"

In [4]:
# 1. Data Loading

In [5]:
import pandas as pd

customers = pd.read_csv(f"{base_path}/olist_customers.csv")
orders = pd.read_csv(f"{base_path}/olist_orders.csv")
order_items = pd.read_csv(f"{base_path}/olist_order_items.csv")
payments = pd.read_csv(f"{base_path}/olist_order_payments.csv")
reviews = pd.read_csv(f"{base_path}/olist_order_reviews.csv")
products = pd.read_csv(f"{base_path}/olist_products.csv")
sellers = pd.read_csv(f"{base_path}/olist_sellers.csv")

In [6]:
# ==============================

In [7]:
# 2. Data Cleaning

In [8]:
customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [9]:
customers.to_sql("customers", conn, if_exists="replace", index=False)
orders.to_sql("orders", conn, if_exists="replace", index=False)
order_items.to_sql("order_items", conn, if_exists="replace", index=False)
payments.to_sql("payments", conn, if_exists="replace", index=False)
reviews.to_sql("reviews", conn, if_exists="replace", index=False)
products.to_sql("products", conn, if_exists="replace", index=False)
sellers.to_sql("sellers", conn, if_exists="replace", index=False)

3095

In [10]:
pd.read_sql("SELECT COUNT(*) FROM customers", conn)

,COUNT(*)
0,99441


In [11]:
query = """
SELECT 
    o.order_id,
    o.customer_id,
    o.order_purchase_timestamp,
    o.order_delivered_customer_date,

    oi.product_id,
    oi.seller_id,
    oi.price,
    oi.freight_value,

    p.payment_value,

    r.review_score,

    pr.product_category_name,

    c.customer_city,
    c.customer_state

FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN order_items oi ON o.order_id = oi.order_id
JOIN payments p ON o.order_id = p.order_id
JOIN reviews r ON o.order_id = r.order_id
JOIN products pr ON oi.product_id = pr.product_id
"""

df = pd.read_sql(query, conn)

df.head()

,order_id,customer_id,order_purchase_timestamp,order_delivered_customer_date,product_id,seller_id,price,freight_value,payment_value,review_score,product_category_name,customer_city,customer_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,2017-10-02 10:56:33,2017-10-10 21:25:13,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,29.99,8.72,2.00,4,utilidades_domesticas,sao paulo,SP
1,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,2017-10-02 10:56:33,2017-10-10 21:25:13,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,29.99,8.72,18.12,4,utilidades_domesticas,sao paulo,SP
2,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,2017-10-02 10:56:33,2017-10-10 21:25:13,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,29.99,8.72,18.59,4,utilidades_domesticas,sao paulo,SP
3,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,2018-07-24 20:41:37,2018-08-07 15:27:45,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,118.70,22.76,141.46,4,perfumaria,barreiras,BA
4,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,2018-08-08 08:38:49,2018-08-17 18:06:29,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,159.90,19.22,179.12,5,automotivo,vianopolis,GO


In [12]:
df.isnull().sum()

order_id                            0
customer_id                         0
order_purchase_timestamp            0
order_delivered_customer_date    2471
product_id                          0
seller_id                           0
price                               0
freight_value                       0
payment_value                       0
review_score                        0
product_category_name            1695
customer_city                       0
customer_state                      0
dtype: int64

In [13]:
df = df.dropna(subset=['order_delivered_customer_date'])

In [14]:
# ==============================

In [15]:
# 3. Feature Engineering

In [16]:
df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])
df['order_delivered_customer_date'] = pd.to_datetime(df['order_delivered_customer_date'])

df['delivery_time'] = (
    df['order_delivered_customer_date'] - df['order_purchase_timestamp']
).dt.days

df['is_delayed'] = df['delivery_time'] > 7

In [17]:
# ==============================

In [18]:
# 4. EDA (explanatory Data Analysis)

In [19]:
# a. Rating distribution

In [20]:
df['review_score'].value_counts(normalize=True)

review_score
5    0.574980
4    0.192847
1    0.114211
3    0.084017
2    0.033946
Name: proportion, dtype: float64

In [21]:
# b. Delivery vs Rating

In [22]:
df.groupby('is_delayed')['review_score'].mean()

is_delayed
False    4.328875
True     3.946164
Name: review_score, dtype: float64

In [23]:
# c. Revenue trend

In [24]:
df['month'] = df['order_purchase_timestamp'].dt.to_period('M')
df.groupby('month')['payment_value'].sum()

month
2016-10      61689.11
2016-12         19.62
2017-01     176887.18
2017-02     326512.68
2017-03     500109.46
2017-04     453172.78
2017-05     701196.69
2017-06     587107.80
2017-07     714455.57
2017-08     798796.98
2017-09     998416.50
2017-10    1002145.84
2017-11    1545236.41
2017-12    1009023.21
2018-01    1369787.45
2018-02    1280261.66
2018-03    1429605.09
2018-04    1459554.40
2018-05    1472433.91
2018-06    1281746.42
2018-07    1294444.31
2018-08    1207183.23
Freq: M, Name: payment_value, dtype: float64

In [25]:
# d. Category performance

In [26]:
df.groupby('product_category_name')['payment_value'].sum().sort_values(ascending=False).head(10)

product_category_name
cama_mesa_banho           1707029.45
beleza_saude              1612019.96
informatica_acessorios    1557592.86
moveis_decoracao          1391930.08
relogios_presentes        1380171.71
esporte_lazer             1348500.16
utilidades_domesticas     1067227.12
ferramentas_jardim         807658.18
automotivo                 783016.78
cool_stuff                 738259.78
Name: payment_value, dtype: float64

In [27]:
# 5. INSIGHTS 

In [28]:
df.groupby('delivery_time')['review_score'].mean()

delivery_time
0      3.421053
1      4.431310
2      4.406479
3      4.366550
4      4.340059
         ...   
189    1.000000
191    1.000000
194    3.000000
195    1.000000
208    2.000000
Name: review_score, Length: 143, dtype: float64

In [29]:
# 6. Building Hypothesis

# Hypothesis 1:
## “Delayed orders reduce customer satisfaction”

# Hypothesis 2:
##“High-value orders have higher expectations → lower ratings”

In [30]:
# 7. Statistical Testing

In [31]:
from scipy.stats import ttest_ind

delayed = df[df['is_delayed'] == True]['review_score']
ontime = df[df['is_delayed'] == False]['review_score']

ttest_ind(delayed, ontime)

TtestResult(statistic=np.float64(-46.35790384720764), pvalue=np.float64(0.0), df=np.float64(114856.0))

In [32]:
## 📊 Hypothesis -1 Testing Result

## We tested whether delayed deliveries impact customer satisfaction.

# - Average rating (On-time): 4.18  
# - Average rating (Delayed): 3.94  
# - p-value: < 0.05  

### ✅ Conclusion:
#The difference in ratings is statistically significant.  
#Delayed deliveries lead to lower customer satisfaction.

#This highlights the importance of delivery performance in shaping customer experience.

In [33]:
df['high_value'] = df['payment_value'] > df['payment_value'].median()

df.groupby('high_value')['review_score'].mean()

high_value
False    4.175983
True     3.984790
Name: review_score, dtype: float64

In [34]:
from scipy.stats import ttest_ind

high = df[df['high_value'] == True]['review_score']
low = df[df['high_value'] == False]['review_score']

ttest_ind(high, low)

TtestResult(statistic=np.float64(-24.100790949735163), pvalue=np.float64(5.111521849242378e-128), df=np.float64(114856.0))

In [35]:
### Hypothesis 2: High-value orders have higher expectations → lower ratings

#We tested whether customers placing higher-value orders tend to give lower ratings.

# - Average rating (Low-value orders): 4.17  
# - Average rating (High-value orders): 3.98  
# - p-value: < 0.05  

### ✅ Conclusion:
#The difference in ratings is statistically significant. High-value orders tend to receive lower ratings, possibly due to higher customer expectations.
#This suggests that customers spending more may expect better service quality and are more critical when expectations are not met.

In [36]:
## 📊 Key Insights

In [37]:
# 1. Delivery performance is a critical driver of customer satisfaction. Delayed orders consistently receive lower ratings.

In [38]:
# 2. High-value customers have stricter expectations and tend to give lower ratings, indicating a need for premium service for higher-paying customers.

In [39]:
# 🚀 Actionables
# 1. Improve delivery logistics to minimize delays and enhance customer satisfaction.  
# 2. Provide accurate delivery timelines to manage customer expectations.  
# 3. Offer premium services (faster shipping, better support) for high-value customers.  
# 4. Monitor seller performance and enforce service quality standards.

In [40]:
## 🔍 Further More Exploration

# Q. Does delivery delay impact repeat customer behavior and long-term retention?

# Understanding this **relationship can help the business** design better customer retention strategies.

In [41]:
df.to_csv("final_dataset.csv", index=False)